# Mean-Variance Portfolio Optimization with cuOpt

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📘 Optional Notebook</h4>
We now support solving <strong>Quadratic Programs (QP)</strong>, enabling standard <strong>Markowitz Mean-Variance</strong> portfolio optimization directly on the GPU.

This notebook shows how the same framework extends to QP problems with a quadratic objective with variance as the risk measure <code>w'Σw</code>.
</div>


## Table of Contents

1. [Setup](#setup) — GPU check & imports
2. [Mathematical Formulation](#introduction) — The Markowitz QP
3. [Solve with cuOpt](#solve) — Data, parameters & GPU optimization
4. [Results](#analyze) — Portfolio visualization & statistics
5. [CPU Solver Comparison](#cpu-comparison) — Clarabel vs cuOpt


---

<a id='setup'></a>
## 1. Setup


In [ ]:
!nvidia-smi

In [ ]:
import os

import numpy as np
import cvxpy as cp

from cufolio import mean_variance_optimizer, utils
from cufolio.mean_variance_parameters import MeanVarianceParameters
from cufolio.settings import ApiSettings, ReturnsComputeSettings

---

<a id='introduction'></a>
## 2. Mathematical Formulation

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Mean-Variance (Markowitz) Optimization</h4>
Mean-Variance optimization, introduced by Harry Markowitz in 1952, is the foundation of Modern Portfolio Theory. The objective trades off expected return against portfolio variance:

\begin{align*}
\text{maximize } \quad &\mu^\top w - \lambda_{\text{risk}} \cdot w^\top \Sigma w\\
\text{subject to} \quad 
& \sum_i w_i + c = 1 \quad \text{(self-financing)}\\
& w_i^{\text{min}} \leq w_i \leq w_i^{\text{max}}, \forall i \quad \text{(concentration)}\\
& c^{\text{min}} \leq c \leq c^{\text{max}} \quad \text{(cash)}\\
& L = \|w\|_1 \leq L^{\text{limit}} \quad \text{(leverage)}\\
& T = \|w - w_{\text{prev}}\|_1 \leq T^{\text{limit}} \quad \text{(turnover)}
\end{align*}

This is a **Quadratic Program (QP)** due to the $w^\top \Sigma w$ term. cuOpt 26.02 introduces QP support via a GPU-accelerated barrier solver.
</div>


---

<a id='solve'></a>
## 3. Solve with cuOpt

<div style="background-color: #e8f5e9; border-left: 6px solid #388e3c; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 End-to-End: Data → Parameters → GPU Solve</h4>
We load S&P 500 price data, compute log-returns, define portfolio constraints, and solve the QP on the GPU using cuOpt.
</div>

<div style="background-color: #fff8e1; border-left: 6px solid #f9a825; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Disclaimer</h4>
Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
</div>


In [ ]:
dataset_name = "sp500"
data_path = f"../data/stock_data/{dataset_name}.csv"

if not os.path.exists(data_path):
    utils.download_data(data_path)

returns_dict = utils.calculate_returns(
    data_path,
    {"name": "recent", "range": ("2022-01-01", "2024-01-01")},
    ReturnsComputeSettings(return_type="LOG", freq=1),
)

print(f"Assets: {len(returns_dict['tickers'])}  |  "
      f"Observations: {len(returns_dict['returns'])}  |  "
      f"Covariance: {returns_dict['covariance'].shape}")


In [ ]:
mean_variance_params = MeanVarianceParameters(
    w_min={"NVDA": 0.1, "others": -0.3},
    w_max={"NVDA": 0.6, "others": 0.4},
    c_min=0.0,
    c_max=0.2,
    L_tar=1.6,
    risk_aversion=1.0,
)


### Solve with CVXPY API

In [ ]:
mean_variance_problem = mean_variance_optimizer.MeanVariance(
    returns_dict=returns_dict,
    mean_variance_params=mean_variance_params,
    api_settings=ApiSettings(api="cvxpy"),
)

gpu_results, gpu_portfolio = mean_variance_problem.solve_optimization_problem(
    solver_settings={"solver": cp.CUOPT, "time_limit": 60, "verbose": True}
)

### Solve using cuOpt Python API

In [ ]:
mean_variance_problem = mean_variance_optimizer.MeanVariance(
    returns_dict=returns_dict,
    mean_variance_params=mean_variance_params,
    api_settings=ApiSettings(api="cuopt_python"),
)

gpu_results, gpu_portfolio = mean_variance_problem.solve_optimization_problem(
    solver_settings={"time_limit": 60, "log_to_console": True}
)

---

<a id='analyze'></a>
## 4. Results


In [ ]:
ax = gpu_portfolio.plot_portfolio(show_plot=True, min_percentage=1)


In [ ]:
weights = gpu_portfolio.weights

expected_return = np.dot(returns_dict["mean"], weights)
portfolio_variance = weights @ returns_dict["covariance"] @ weights
portfolio_std = np.sqrt(portfolio_variance)
sharpe_ratio = expected_return / portfolio_std if portfolio_std > 0 else 0

print("Portfolio Statistics:")
print(f"  Expected Return: {expected_return:.4%}")
print(f"  Volatility (Std): {portfolio_std:.4%}")
print(f"  Variance: {portfolio_variance:.6f}")
print(f"  Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"  Cash Allocation: {gpu_portfolio.cash:.2%}")
print(f"  Leverage: {np.sum(np.abs(weights)):.4f}")
print(f"  Number of positions: {np.sum(np.abs(weights) > 0.001)}")


---

<a id='cpu-comparison'></a>
## 5. CPU Solver Comparison

<div style="background-color: #fff3e0; border-left: 6px solid #f57c00; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🔄 GPU vs CPU</h4>
Solve the same Mean-Variance problem using <strong>Clarabel</strong> (a CPU-based interior-point solver) and compare the results against the GPU solution above. For small portfolios the GPU advantage in solve time alone may be modest, but running the entire pipeline on GPU — data preprocessing, returns forecasting, scenario generation, and optimization — eliminates costly CPU↔GPU data transfers and delivers end-to-end speedups that grow with problem size.
</div>

In [ ]:
cpu_problem = mean_variance_optimizer.MeanVariance(
    returns_dict=returns_dict,
    mean_variance_params=mean_variance_params,
    api_settings=ApiSettings(api="cvxpy"),
)

cpu_results, cpu_portfolio = cpu_problem.solve_optimization_problem(
    solver_settings={"solver": cp.CLARABEL}
)

In [ ]:
utils.compare_results(gpu_results.to_dict(), cpu_results.to_dict())

---

## Summary

<div style="background-color: #e8f5e9; border-left: 6px solid #388e3c; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">✅ Key Takeaway</h4>
<strong>cuOpt >= 26.02</strong> adds QP support, enabling GPU-accelerated Markowitz Mean-Variance optimization alongside the existing LP-based CVaR solver. The same cuFOLIO framework handles both problem types — simply choose the appropriate optimizer class.

### Mean-Variance vs CVaR

| Aspect | Mean-Variance | CVaR |
|--------|---------------|------|
| Risk Measure | Variance (symmetric) | Tail risk (asymmetric) |
| Problem Type | QP (Quadratic) | LP (Linear) |
| Data Needed | Mean + Covariance | Scenarios |
| Distribution | Assumes normal | Distribution-free |

</div>
